# 34b — Common/difference decomposition through four concepts

Notebook 34 established the decomposition triptych for pairs and triples:

- `gds-blend`: difference-only guide, $\Pi((I-MM^T)\sum_i F_i)$;
- `blend`: full guide, $\Pi(\sum_i F_i)$;
- `common-blend`: common-only guide, $\Pi(MM^T\sum_i F_i)$.

Notebook 39 already generated the matching full-blend and common-only arms for
30 nested quads. This extension runs only the missing difference-only arm with
the same bf16 Llama model, greedy survivor rule, `k=4`, 24 steps, prompts, and
concept groups. It then analyzes **joint expression only** across `n=2,3,4`.
No rho-stratified analysis is performed. Evaluation is against the original,
unprojected concept frames. Confidence intervals resample concept groups, not
individual prompt records.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.experiments.notebook_helpers import llama_chat, read_jsonl
from frames.linalg.gds import gds_blend
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS = 3
K = 4
STEPS = 24
COMMON_THRESHOLD = 1.05
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"

QUADS_PATH = Path("resources/39_quads.json")
LOG_PATH = Path("resources/34b_gds_blend_n4.jsonl")
SUMMARY_PATH = Path("resources/34b_joint_expression_summary.csv")
PAIRWISE_PATH = Path("resources/34b_joint_expression_pairwise.csv")
FIGURE_PATH = Path("resources/34b_joint_expression_n2_n4.png")

QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]
prompts = [llama_chat(user, assistant) for user, assistant in QUESTIONS]
quads = json.loads(QUADS_PATH.read_text())["quads"]
assert len(quads) == 30
print(f"{len(quads)} quads; missing arm writes to {LOG_PATH}")

In [ ]:
def synsets_from_config(config):
    return tuple(config[key] for key in ("a", "b", "c", "d") if config.get(key))


existing = read_jsonl(LOG_PATH) if LOG_PATH.exists() and LOG_PATH.stat().st_size else []
completed = {(synsets_from_config(row["config"]), row["prompt"]) for row in existing}
missing_count = 30 * len(prompts) - len(completed)
print(f"resume gate: {len(completed)} complete, {missing_count} missing records")

if missing_count:
    fur = FrameUnembeddingRepresentation.from_model_id(
        device_map="auto", id=MODEL_ID, torch_dtype=torch.bfloat16
    )
    harness = EvaluationHarness(
        fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
    )
    baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)
    names = sorted({q[key] for q in quads for key in ("a", "b", "c", "d")})
    concepts = {}
    for name in names:
        concept = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
        concepts[name] = Concept(
            synset=concept.synset, tensor=concept.tensor.to(fur.model.device)
        )
    print(f"{len(concepts)} concepts built")

    for quad in quads:
        synsets = tuple(quad[key] for key in ("a", "b", "c", "d"))
        missing = [
            index
            for index, prompt in enumerate(prompts)
            if (synsets, prompt) not in completed
        ]
        if not missing:
            continue
        group_concepts = [concepts[name] for name in synsets]
        frames = torch.cat(Concept._pad_to_common_rank(group_concepts))
        guide_tensor = gds_blend(frames, common_threshold=COMMON_THRESHOLD)
        if int(torch.linalg.matrix_rank(guide_tensor.float())) == 0:
            raise RuntimeError(f"difference-only guide collapsed for {synsets}")
        guide = Concept(
            synset=" + ".join(synsets) + " (gds)",
            tensor=guide_tensor.unsqueeze(0),
        )
        run_prompts = [prompts[index] for index in missing]
        torch.cuda.empty_cache()
        texts, probe = fur.generate_with_topk_guide(
            run_prompts, guide=guide, k=K, steps=STEPS
        )
        harness.evaluate(
            run_prompts,
            texts,
            list(synsets),
            config={
                "model": MODEL_KEY,
                "method": "gds-blend",
                "n": 4,
                "steps": STEPS,
                "common_threshold": COMMON_THRESHOLD,
                "source_groups": "39_quads",
                **dict(zip(("a", "b", "c", "d"), synsets)),
            },
            probe=probe,
            baseline_texts=[baseline_texts[index] for index in missing],
            concepts=dict(zip(synsets, group_concepts)),
            angles=False,
        )
        print(f"{' + '.join(name.split('.')[0] for name in synsets)}: done")

fresh = read_jsonl(LOG_PATH)
assert len(fresh) == 300, f"expected 300 n=4 records, found {len(fresh)}"

In [ ]:
def joint_rows(path, methods, ns):
    rows = []
    for record in read_jsonl(path):
        config = record["config"]
        if config["method"] not in methods or config["n"] not in ns:
            continue
        synsets = synsets_from_config(config)
        deltas = [record["expression"][name]["delta"] for name in synsets]
        rows.append(
            {
                "method": config["method"],
                "n": config["n"],
                "group": " | ".join(synsets),
                "joint": min(deltas),
                "all_positive": min(deltas) > 0,
            }
        )
    return pd.DataFrame(rows)


results = pd.concat(
    [
        joint_rows("resources/33_scaling3_bf16.jsonl", {"blend", "gds-blend"}, {2, 3}),
        joint_rows("resources/34_common_blend.jsonl", {"common-blend"}, {2, 3}),
        joint_rows("resources/39_scaling4.jsonl", {"blend", "common-blend"}, {4}),
        joint_rows(LOG_PATH, {"gds-blend"}, {4}),
    ],
    ignore_index=True,
)

expected_runs = {2: 100, 3: 300, 4: 300}
for n, count in expected_runs.items():
    for method in ("gds-blend", "blend", "common-blend"):
        actual = len(results[(results["n"] == n) & (results["method"] == method)])
        assert actual == count, (n, method, actual)
    sets = [
        set(results[(results["n"] == n) & (results["method"] == method)]["group"])
        for method in ("gds-blend", "blend", "common-blend")
    ]
    assert sets[0] == sets[1] == sets[2], f"group mismatch at n={n}"
print(f"analysis gate: {len(results)} records")

In [ ]:
def bootstrap_ci(values, seed, draws=10_000):
    values = np.asarray(values, dtype=float)
    generator = np.random.default_rng(seed)
    sampled = generator.choice(values, size=(draws, len(values)), replace=True).mean(
        axis=1
    )
    return np.quantile(sampled, [0.025, 0.975])


group_means = results.groupby(["method", "n", "group"], as_index=False).agg(
    joint=("joint", "mean"), all_positive=("all_positive", "mean")
)
summary_rows = []
for index, ((method, n), part) in enumerate(group_means.groupby(["method", "n"])):
    low, high = bootstrap_ci(part["joint"], seed=3400 + index)
    records = results[(results["method"] == method) & (results["n"] == n)]
    summary_rows.append(
        {
            "method": method,
            "n": n,
            "joint": records["joint"].mean(),
            "joint_ci_low": low,
            "joint_ci_high": high,
            "all_positive": records["all_positive"].mean(),
            "n_groups": len(part),
            "n_runs": len(records),
        }
    )
summary = pd.DataFrame(summary_rows).sort_values(["n", "method"])
summary.to_csv(SUMMARY_PATH, index=False)
summary.round(4)

In [ ]:
paired_rows = []
for n in (2, 3, 4):
    pivot = group_means[group_means["n"] == n].pivot(
        index="group", columns="method", values="joint"
    )
    for method in ("gds-blend", "common-blend"):
        deltas = pivot[method] - pivot["blend"]
        low, high = bootstrap_ci(deltas, seed=34_000 + n * 10 + len(paired_rows))
        paired_rows.append(
            {
                "method": method,
                "reference": "blend",
                "n": n,
                "mean_difference": deltas.mean(),
                "ci_low": low,
                "ci_high": high,
                "win_rate": (deltas > 0).mean(),
                "n_groups": len(deltas),
            }
        )
pairwise = pd.DataFrame(paired_rows)
pairwise.to_csv(PAIRWISE_PATH, index=False)
pairwise.round(4)

In [ ]:
order = ["gds-blend", "blend", "common-blend"]
labels = {
    "gds-blend": "difference only",
    "blend": "full blend",
    "common-blend": "common only",
}
colors = {"gds-blend": "navy", "blend": "tab:blue", "common-blend": "tab:red"}
x = np.arange(3)
width = 0.24
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

for offset, method in zip((-width, 0, width), order):
    part = summary[summary["method"] == method].set_index("n").loc[[2, 3, 4]]
    values = part["joint"].to_numpy()
    errors = np.vstack(
        [
            values - part["joint_ci_low"].to_numpy(),
            part["joint_ci_high"].to_numpy() - values,
        ]
    )
    axes[0].bar(
        x + offset,
        values,
        width,
        yerr=errors,
        capsize=3,
        color=colors[method],
        label=labels[method],
    )
axes[0].set_xticks(x, ["n=2", "n=3", "n=4"])
axes[0].set_ylabel("mean joint expression gain")
axes[0].axhline(0, color="gray", lw=0.8)
axes[0].set_title("decomposition triptych")
axes[0].legend(fontsize=8)

for method, marker in (("gds-blend", "o"), ("common-blend", "s")):
    part = pairwise[pairwise["method"] == method].set_index("n").loc[[2, 3, 4]]
    values = part["mean_difference"].to_numpy()
    errors = np.vstack(
        [values - part["ci_low"].to_numpy(), part["ci_high"].to_numpy() - values]
    )
    axes[1].errorbar(
        [2, 3, 4],
        values,
        yerr=errors,
        marker=marker,
        capsize=4,
        color=colors[method],
        label=f"{labels[method]} - full blend",
    )
axes[1].set_xticks([2, 3, 4])
axes[1].set_xlabel("concurrent concepts n")
axes[1].set_ylabel("paired group-level joint difference")
axes[1].axhline(0, color="gray", lw=0.8)
axes[1].set_title("operator effect relative to full blend")
axes[1].legend(fontsize=8)

plt.suptitle("GDS decomposition and joint expression through n=4")
plt.tight_layout()
plt.savefig(FIGURE_PATH, dpi=180, bbox_inches="tight")
plt.show()
print(f"wrote {SUMMARY_PATH}, {PAIRWISE_PATH}, and {FIGURE_PATH}")